# 5.6 算子入图实践

## 1. 背景简介

算子入图是指将 Ascend C 自定义算子接入图执行链路，使其脱离逐算子即时下发、转为按图组织与复用执行，从而获得相比 Eager 逐算子下发更高的执行效率。本小节从算子实现的编写与验证出发，围绕算子入图的两种典型方式，给出可复现的端到端实践路径：

- **算子直调工程**：编写包含 `op_kernel`、`op_host`、`op_extension`、`scripts` 的 Kernel 直调工程，先在本地完成编译、直调与精度自验证，为后续图化接入提供可用的算子实现。
- **AclGraph**：面向 Host 调度开销优化的轻量化方案，采用「流任务捕获 + 模型化执行」，全程无需修改算子本身的实现逻辑，适用于重复执行的固定任务序列。

两种方式的对比如下表所示。

| 接入方式 | 核心机制 | 是否需要改造算子 | 典型场景 |
| --- | --- | --- | --- |
| 算子直调工程 | 编写 Kernel 直调工程 → 本地编译 → 直接下发执行 | 需要：实现 Kernel 计算逻辑与 Tiling 切分 | 算子开发与精度自验证 |
| AclGraph | 流任务捕获录制 → 模型化批量异步执行 → 资源销毁 | 不需要：算子实现不变，仅改变下发方式 | 固定任务序列的重复下发 |

> 实践一以 Softmax 为例，讲解如何实现算子直调工程并完成精度自验证；实践二沿用逐元素加法（`AddCustom`）作为教学示例算子，便于聚焦接入链路本身。实际业务中可将示例算子替换为 Softmax、Matmul 等，详见本文第 5 节「课后练习」。

## 2. 使用约束

- 需要在已安装匹配版本 CANN、PyTorch 与 `torch_npu` 的昇腾 NPU 环境中运行，CANN 侧需具备可编译 Ascend C 算子的完整工具链。
- 实践二（AclGraph）依赖 `aclmdlRI` 系列接口，要求 CANN ≥ 9.1.0，且产品型号在支持范围内（Ascend 950PR/950DT、Atlas A3、Atlas A2 系列）。
- 待接入算子须已具备可用的 Ascend C NPU 实现（Kernel）与算子原型（`REG_OP`）。
- 实践一需自行编写 Kernel 直调工程并完成精度自验证；实践二侧重 AclGraph 捕获与执行，两者相互独立，可按需选择。
- 示例用于教学演示，算子的 shape、dtype 约束以各示例源码为准；生产接入需补充完整的类型、形状与边界校验。

## 3. 实践一：实现算子 Kernel 直调工程并完成精度自验证

算子接入图执行链路的前提，是先有一个可编译、可运行且精度正确的 Ascend C 算子实现。本节以 Softmax 为例，从零编写一个 Kernel 直调工程，并完成精度自验证。

### 3.1 工程模板与场景选择

官方 Kernel 直调工程模板位于 `cannbot-skills` 仓库的 `ops/ascendc-direct-invoke-template/references/` 目录下，按场景分为三类：

| 场景 | 模板路径 | 适用算子 |
| --- | --- | --- |
| Vector | `references/add_custom/` | 逐元素与规约类算子，如 Add、Softmax |
| Blaze Matmul | `references/matmul_blaze_template/` | 矩阵乘类算子，如 Matmul |
| Kirin Vector | `references/kirin_add_template/` | Kirin 平台 Vector 算子 |

Softmax 沿最后一维做行内规约，属于 Vector 场景，工程以 `references/add_custom/` 为骨架实现。

### 3.2 实现步骤

模板中需要实现的位置均以 `[MODIFY]` 标记，按标记逐项完成即可：

1. **创建工程目录**：将 `references/add_custom/` 复制为 `<op>_custom/` 工程目录。
2. **统一算子名**：将工程中的 `add_custom` 统一替换为算子名（如 `softmax_custom`），并同步修改 `CMakeLists.txt` 中的可执行文件与动态库目标名、`run.sh` 的 `OP_NAME`、`scripts/test_torch.py` 的 `SO_NAME` 与 `OP_NAME`。注意模板 `run.sh` 里的 `OP_NAME` 默认值是 `add`，全局替换 `add_custom` 覆盖不到它，需手动改成实际的可执行文件名。
3. **改写 Kernel**：在 `op_kernel/<op>_kernel.asc` 中将 `AscendC::Add` 替换为算子的计算逻辑。
4. **改写 Tiling**：在 `op_kernel/<op>_tiling.h` 中定义 Tiling 结构体，并在 `op_host/<op>.asc` 中完成切分计算。
5. **调整输入输出数量**：同步修改 Kernel 入口参数、host 侧 `KernelCall` 与 `op_extension/<op>_torch.cpp` 中的参数个数。

Softmax 的 Kernel 计算流程如下（每行独立归一化）：

| 步骤 | API | 说明 |
| --- | --- | --- |
| 1 | `AscendC::ReduceMax` | 求每行最大值，用于数值稳定 |
| 2 | `AscendC::Adds` | 每行减去最大值 |
| 3 | `AscendC::Exp` | 逐元素取指数 |
| 4 | `AscendC::ReduceSum` | 求每行指数和 |
| 5 | `AscendC::Muls` | 乘以指数和的倒数，完成归一化 |

### 3.3 工程目录结构

```text
<op>_custom/
├── CMakeLists.txt              # 双目标：可执行文件 + lib<op>_ops.so
├── README.md                   # 算子说明与运行方式
├── run.sh                      # 一键编译与直调运行
├── op_kernel/
│   ├── <op>_tiling.h           # Tiling 结构体定义
│   └── <op>_kernel.asc         # Kernel 实现
├── op_host/
│   ├── <op>.asc                # host 侧 Tiling 计算与 KernelCall
│   └── data_utils.h
├── op_extension/
│   ├── <op>_torch.cpp          # PyTorch 接入封装
│   ├── register.cpp            # schema 与 Meta 注册
│   └── ops.h
└── scripts/
    ├── gen_data.py             # 生成输入与 golden
    ├── golden.py               # CPU 参考实现
    ├── verify_result.py        # 直调结果与 golden 比对
    └── test_torch.py           # PyTorch 侧调用验证
```

### 3.4 精度自验证流程

工程实现完成后，按「编译 → 生成测试数据 → 直调运行 → 结果比对」的流程验证。模板 `run.sh` 已把整条链路串好，在工程根目录执行即可（末段 PyTorch 通路的路径注意事项见下方提示）：

```bash
# 一键完成：编译 → 生成测试数据 → 直调运行 → 精度比对
bash run.sh

# 复用已有编译产物，跳过编译
bash run.sh --skip-build
```

如需分步执行，注意 `gen_data.py` 按相对路径读写，数据与产物都落在 `build` 目录下：

```bash
mkdir -p build && cd build
cmake .. && make -j4
python3 ../scripts/gen_data.py
./softmax_custom
python3 ../scripts/verify_result.py output/output.bin output/golden.bin
```

PyTorch 通路验证需回到工程根目录执行，脚本按 `build/lib<op>_ops.so` 加载算子库：

```bash
python3 scripts/test_torch.py
```

> `verify_result.py` 必须显式传入输出与 golden 两个路径，不传参只会打印 Usage 并以非零码退出。上述步骤全部通过后，该算子实现即可作为后续图化接入的基础。

> `test_torch.py` 按 `build/<SO_NAME>` 加载算子库，必须在工程根目录执行。模板 `run.sh` 是在 `build` 目录内调用该脚本的，相对路径会解析成 `build/build/<SO_NAME>`，脚本随即打印 `ERROR: build/lib<op>_ops.so not found` 并非零退出（`run.sh` 开启 `set -e`，整条流程随之终止）。遇到该提示时，回到工程根目录重跑 `python3 scripts/test_torch.py` 即可，前序的编译、造数、直调运行与精度比对结果均有效、无需重复执行。

> 工程模板与完整样例可参考：<https://gitcode.com/cann/cannbot-skills/tree/master/ops/ascendc-direct-invoke-template>

In [ ]:
import re
from pathlib import Path

# Kernel 直调工程目录与算子名（按实际工程修改）
PROJ = Path("softmax_custom")
OP = "softmax_custom"

# Softmax 工程应包含的文件（以 op 名 softmax_custom 为例）
EXPECTED = [
    "CMakeLists.txt",
    "README.md",
    "run.sh",
    "op_kernel/softmax_custom_tiling.h",
    "op_kernel/softmax_custom_kernel.asc",
    "op_host/softmax_custom.asc",
    "op_host/data_utils.h",
    "op_extension/softmax_custom_torch.cpp",
    "op_extension/register.cpp",
    "op_extension/ops.h",
    "scripts/gen_data.py",
    "scripts/golden.py",
    "scripts/verify_result.py",
    "scripts/test_torch.py",
]

if PROJ.is_dir():
    missing = [f for f in EXPECTED if not (PROJ / f).is_file()]
    print("工程目录：", PROJ.resolve())
    print("应含文件数：", len(EXPECTED), "缺失：", missing if missing else "无")

    # run.sh 的 OP_NAME 需与 CMakeLists.txt 中的可执行目标名一致
    run_sh = PROJ / "run.sh"
    if run_sh.is_file():
        m = re.search(
            r'^\s*OP_NAME\s*=\s*"([^"]+)"',
            run_sh.read_text(encoding="utf-8", errors="ignore"),
            re.M,
        )
        if m:
            state = "一致" if m.group(1) == OP else "不一致，请改为 %s" % OP
            print("run.sh OP_NAME：%s（%s）" % (m.group(1), state))
else:
    print("未检测到工程目录 %s，请先按上述结构实现工程：" % PROJ)
    for f in EXPECTED:
        print("  ", f)

# 精度自验证流程（以下命令均在工程根目录执行）
print()
print("运行流程：")
for i, (cmd, note) in enumerate(
    [
        ("bash run.sh", "编译 → 生成测试数据 → 直调运行 → 精度比对"),
        ("bash run.sh --skip-build", "复用已有编译产物，跳过编译"),
        ("python3 scripts/test_torch.py", "PyTorch 通路验证"),
    ],
    1,
):
    print("  %d. %s    # %s" % (i, cmd, note))

## 4. 实践二：Ascend C 算子接入 AclGraph

### 4.1 背景与方案

以 PyTorch 为代表的主流 AI 框架默认采用 Eager 即时执行模式：无需预构建计算图，算子逐一下发、边下发边执行，优势是执行结果即时可见、调试便捷；但逐算子调度会产生持续的 Host 侧下发开销，在深度性能优化中会成为不可忽视的性能瓶颈。

针对 Host 开销问题，ACL 提供了「流任务捕获 + 模型化执行」的 Graph 方案，将计算任务下沉至 Device 端消解调度开销，核心流程与价值如下：

- **任务捕获**：在 `aclmdlRICaptureBegin` 与 `aclmdlRICaptureEnd` 区间内，指定 Stream 上的任务暂不执行，统一被捕获并暂存至模型运行实例中。
- **批量执行**：通过 `aclmdlRIExecuteAsync` 接口一次性触发所有暂存任务执行；针对需重复执行的场景，无需重复下发任务，仅需多次调用执行接口，大幅削减 Host 侧重复调度的开销。
- **资源释放**：使用完毕后调用 `aclmdlRIDestroy` 销毁运行实例，回收对应资源。

示意流程：**捕获录制（Begin → 下发任务 → End）→ 批量执行（ExecuteAsync × N）→ 资源销毁（Destroy）**。

捕获区间内只允许异步下发类任务：`aclrtSynchronizeStream`、`aclrtEventQuery` 等同步或查询接口会打破「只记录、不执行」的前提，必须放在区间之外；而 `aclrtMemcpyAsync`、`aclrtRecordEvent`、`aclrtStreamWaitEvent` 等异步接口可以正常入图。

### 4.2 AclGraph 调用示例

```cpp
// 前置：已创建 mainStream，并完成 host / device 内存申请
aclmdlRICaptureBegin(mainStream, ACL_MODEL_RI_CAPTURE_MODE_GLOBAL);
aclrtMemcpyAsync(xDevice, totalByteSize, xHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE, mainStream);
aclrtMemcpyAsync(yDevice, totalByteSize, yHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE, mainStream);
add_custom<blockLength><<<numBlocks, 0, mainStream>>>(xDevice, yDevice, zDevice);
aclrtMemcpyAsync(zHost, totalByteSize, zDevice, totalByteSize, ACL_MEMCPY_DEVICE_TO_HOST, mainStream);
aclmdlRICaptureEnd(mainStream, &modelRI);
aclmdlRIDebugJsonPrint(modelRI, "./modelRI_single_stream.json", 0);

// 捕获完成后重复执行，无需再次逐条下发任务
for (int i = 0; i < 5; ++i) {
    aclmdlRIExecuteAsync(modelRI, mainStream);
    aclrtSynchronizeStream(mainStream);
}

aclmdlRIDestroy(modelRI);
```

需要说明的是，AclGraph 接入**无需修改算子本身的实现逻辑**，只需将算子的下发动作置于捕获区间内，即可完成捕获与复用。

> 完整样例请参考：<https://gitcode.com/cann/asc-devkit/blob/master/examples/01_simd_cpp_api/02_features/00_framework/04_aclgraph/README.md>。该样例提供 `SCENARIO_NUM=1`（单流捕获）与 `SCENARIO_NUM=2`（双流捕获）两种场景，双流场景通过 `aclrtRecordEvent` / `aclrtStreamWaitEvent` 建立跨流依赖。

## 5. 课后练习

### 一、单选题

1. Kernel 直调工程中，算子 Kernel 入口函数所在的文件是？
- A. `op_host/<op>.asc`
- B. `op_kernel/<op>_kernel.asc`
- C. `op_extension/<op>_torch.cpp`
- D. `scripts/golden.py`

2. AclGraph 中，标记捕获区间结束的接口是？
- A. `aclmdlRICaptureBegin`
- B. `aclmdlRIExecuteAsync`
- C. `aclmdlRIDestroy`
- D. `aclmdlRICaptureEnd`

3. AclGraph 中，一次性触发所有暂存任务执行的接口是？
- A. `aclmdlRICaptureEnd`
- B. `aclrtSynchronizeStream`
- C. `aclmdlRIExecuteAsync`
- D. `aclmdlRIDebugJsonPrint`

4. 直调工程中，用于逐元素比对本算子输出与 golden 结果的脚本是？
- A. `scripts/gen_data.py`
- B. `scripts/golden.py`
- C. `scripts/verify_result.py`
- D. `scripts/test_torch.py`

### 二、多选题

5. 关于 AclGraph 的说法，正确的有哪些？
- A. 采用「流任务捕获 + 模型化执行」的设计思路
- B. 目的是消减算子逐一下发的 Host 调度开销
- C. 接入时需要修改算子本身的实现逻辑
- D. 重复执行时无需重复下发任务，仅需多次调用执行接口

6. 关于 Kernel 直调工程的说法，正确的有哪些？
- A. 工程包含 `op_kernel`、`op_host`、`op_extension`、`scripts` 等目录
- B. 需按模板中的 `[MODIFY]` 标记替换算子名、Tiling 结构体与计算逻辑
- C. 执行 `bash run.sh` 即可完成编译并直调运行 Kernel
- D. 生成的工程只能在本地直调运行，无法在 PyTorch 中调用

7. 关于算子精度自验证流程，说法正确的有哪些？
- A. `scripts/gen_data.py` 负责生成输入数据与 golden 结果
- B. `scripts/verify_result.py` 将直调输出与 golden 逐元素比对
- C. 通过 `torch.ops.load_library` 加载 `lib<op>_ops.so` 后，可在 PyTorch 中调用该算子
- D. 直调验证通过后，接入 AclGraph 仍需修改算子 Kernel 实现

### 三、编程题

8. 参考实践一，实现 Softmax（或 Matmul）算子的 Kernel 直调工程，完成编译、直调与精度自验证；再参考实践二，将其接入 AclGraph，并完成数值正确性验证。

### 四、简答题

9. 结合本章两个实践，说明算子直调工程与 AclGraph 两种方式在「是否需要改造算子」「适用场景」「性能收益来源」上的差异。

**运行以下代码单元查看参考答案与解析。**

In [ ]:
from pathlib import Path

answer = Path("answer/05.06_answer.txt")
print(answer.read_text(encoding="utf-8"))